# Test graph

In [1]:

import gc
import os
import re
import json
from pathlib import Path
from typing import Any
import random
import pandas as pd

from dotenv import load_dotenv

# load env
load_dotenv()


True

In [10]:
DATA_DIR = Path("../data")
print(f"DATA_DIR: {DATA_DIR.resolve()}")

GRAPH_DIR = DATA_DIR / "graph"


DATA_DIR: C:\52300070_TMTHUYEN\LEARN_MORE\Apps\datn-legal-ai\backend\data


In [2]:
from src.graph.builder_graph import BuilderGraph
from src.graph.graph_repository import LegalGraphRepository
from src.graph.graph_service import LegalGraphService
from src.graph.neo4j_client import Neo4jClient
from src.utils.logger import setup_logger

print("Starting test_venv.py..." + os.environ.get("MODE", ""))

class TestGraph:
    def __init__(self, service: LegalGraphService):
        self.logger = setup_logger(name="test_venv", filename="test_venv.log")
        self.logger.info("TestVenv initialized.")
        self.builder_graph = BuilderGraph()
        self.legal_service = service

    def test_import(self):
        self.logger.info("Ok import of modules...")
        self.builder_graph.add_node("node1", {"data": "value1"})
        
    def test_graph(self):
        self.logger.info("Testing graph service...")
        
        # nodes
        legal_nodes = self.legal_service.get_all_legal_nodes(limit=5000)
        self.logger.info(f"Retrieved {len(legal_nodes)} legal nodes.")
        
        # 10 first nodes
        for i, node in enumerate(legal_nodes[:10]):
            self.logger.info(f"Node {i+1}: {node}")
        
        # relationships
        legal_relationships = self.legal_service.get_all_legal_relationships(limit=5000)
        self.logger.info(f"Retrieved {len(legal_relationships)} legal relationships.")
        
        # 10 first relationships
        for i, relationship in enumerate(legal_relationships[:10]):
            self.logger.info(f"Relationship {i+1}: {relationship}")
        
        gc.collect()  # Force garbage collection to free up memory
        
print(os.environ.get("NEO4J_URI"))
neo4j_client = Neo4jClient(
    uri=os.environ.get("NEO4J_URI"),
    username=os.environ.get("NEO4J_USERNAME"),
    password=os.environ.get("NEO4J_PASSWORD"),
    database=os.environ.get("NEO4J_DATABASE")
)
repo = LegalGraphRepository(db=neo4j_client)
service = LegalGraphService(repository=repo)
test_graph = TestGraph(service=service)
test_graph.test_import()
test_graph.test_graph()

2026-10-13 20:57:05 - [INFO] - 1957915430.py - TestVenv initialized.
2026-10-13 20:57:05 - [INFO] - builder_graph.py - BuilderGraph initialized.
2026-10-13 20:57:05 - [INFO] - 1957915430.py - Ok import of modules...
2026-10-13 20:57:05 - [INFO] - builder_graph.py - Node node1 added to the graph.
2026-10-13 20:57:05 - [INFO] - 1957915430.py - Testing graph service...
2026-10-13 20:57:05 - [INFO] - graph_repository.py - Tìm tất cả node có nhãn 'LegalNode', limit=5000, order_by=None...


Starting test_venv.py...development
bolt://localhost:7687


2026-10-13 20:57:07 - [INFO] - graph_repository.py - Tìm thấy 3472 node có nhãn 'LegalNode'.
2026-10-13 20:57:07 - [INFO] - 1957915430.py - Retrieved 3472 legal nodes.
2026-10-13 20:57:07 - [INFO] - 1957915430.py - Node 1: {'verification_scope': 'effective_from verified via NQ 64/2013/QH13 (ItemID 32804); status verified via CSDL Quốc gia VBPL ItemID 32801 (sửa đổi bởi NQ 203/2025/QH15)', 'status_verified_at': '2026-10-07', 'label': 'Document', 'document_id': 'HIEN_PHAP_2013', 'title': 'Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam năm 2013', 'issuer': 'Quốc hội', 'source_file': 'Hien_phap_nam_2013.docx', 'source_url': 'https://vbpl.vn/TW/Pages/vbpq-toanvan.aspx?ItemID=32801', 'number': 'Không số', 'metadata_source': 'DOCX nguồn: van_ban/Hien_phap_nam_2013.docx; effective_from: Nghị quyết 64/2013/QH13, Điều 1; status: CSDL Quốc gia VBPL ItemID 32801 (sửa đổi bởi Nghị quyết 203/2025/QH15)', 'effective_from_source_url': 'https://vbpl.vn/TW/Pages/vbpq-toanvan.aspx?ItemID=32804', 'veri

In [17]:
def save_to_json(data: Any, filename: str | Path):
    """Save data to a JSON file."""
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
    print(f"Data saved to {filename}")

In [37]:
client  = Neo4jClient()                       # đọc NEO4J_URI/USERNAME/PASSWORD/DATABASE từ .env
repo    = LegalGraphRepository(db=client)
service = LegalGraphService(repository=repo)

save_to_json(service.get_all_documents(), GRAPH_DIR / "all_documents.json")
save_to_json(service.get_all_legal_nodes(limit=10000), GRAPH_DIR / "all_legal_nodes.json")
save_to_json(service.get_node_by_id("BLDS_2015_D27_K1_DIEM_A"), GRAPH_DIR / "BLDS_2015_D27_K1_DIEM_A.json")

save_to_json(
    service.resolve_target_node("BLDS_2015", {"chapter": "III", "article": "27", "clause": "1", "point": "a"}), 
    GRAPH_DIR / "BLDS_2015_resolve_target_node.json"
)
save_to_json(
    service.get_breadcrumbs("BLDS_2015_D27_K1"), 
    GRAPH_DIR / "BLDS_2015_breadcrumbs.json"
)

# service.get_children("BLDS_2015_D27")         # các Khoản của Điều 27
save_to_json(service.get_children("BLDS_2015_D27"), GRAPH_DIR / "BLDS_2015_D27_children.json")
# service.get_descendants("BLDS_2015_P1_CH3", max_depth=5, limit=200)
save_to_json(service.get_descendants("BLDS_2015_P1_CH3", max_depth=5, limit=200), GRAPH_DIR / "BLDS_2015_P1_CH3_descendants.json")
# service.get_related_laws("BLDS_2015_D27_K1")  # REFERENCES/IMPLEMENTS 1-hop
save_to_json(service.get_related_laws("BLDS_2015_D27_K1"), GRAPH_DIR / "BLDS_2015_D27_K1_related_laws.json")
# service.get_document_provenance("BLDS_2015")  # số hiệu, hiệu lực, file nguồn
save_to_json(service.get_document_provenance("BLDS_2015"), GRAPH_DIR / "BLDS_2015_provenance.json")
# service.get_graph_summary()                   # thống kê node/relation theo loại
save_to_json(service.get_graph_summary(), GRAPH_DIR / "graph_summary.json")

2026-10-13 21:27:36 - [INFO] - graph_repository.py - Tìm tất cả node có nhãn 'LegalNode', limit=10000, order_by=None...


Data saved to ..\data\graph\all_documents.json


2026-10-13 21:27:36 - [INFO] - graph_repository.py - Tìm thấy 3472 node có nhãn 'LegalNode'.
2026-10-13 21:27:36 - [INFO] - graph_repository.py - Tìm node theo node_id='BLDS_2015_D27_K1_DIEM_A'...
2026-10-13 21:27:36 - [INFO] - graph_repository.py - Phân giải node phân cấp: doc=BLDS_2015, art=27, k=1, diem=a, ch=III, sec=None
Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `type` does not exist in database `legalai`. Verify that the spelling is correct.', position=<SummaryInputPosition line=5, column=21, offset=178>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 178, 'line': 5, 'column': 21}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}

Data saved to ..\data\graph\all_legal_nodes.json
Data saved to ..\data\graph\BLDS_2015_D27_K1_DIEM_A.json
Data saved to ..\data\graph\BLDS_2015_resolve_target_node.json
Data saved to ..\data\graph\BLDS_2015_breadcrumbs.json
Data saved to ..\data\graph\BLDS_2015_D27_children.json
Data saved to ..\data\graph\BLDS_2015_P1_CH3_descendants.json
Data saved to ..\data\graph\BLDS_2015_D27_K1_related_laws.json
Data saved to ..\data\graph\BLDS_2015_provenance.json
Data saved to ..\data\graph\graph_summary.json


# Truy vấn luật với Graph từ input

In [33]:

import gc
import os
import re
import json
from pathlib import Path
from typing import Any
import random
import pandas as pd

from dotenv import load_dotenv

# load env
load_dotenv()


True

## Cấu hình và đương dẫn file

In [34]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)


ROOT_DIR = Path("../").resolve()
print(f"ROOT_DIR: {ROOT_DIR}")

DATA_DIR = ROOT_DIR / "data"

RETRIEVAL_DIR = DATA_DIR / "retrieval"

QA_DATA_FILE = DATA_DIR / "qa" / "qa_data.json"

TASK_ID = "LEGAL_QUERY_REFERENCE_HIERARCHY_EXTRACTION"

DATA_COLUMNS = ["task_id", "input", "output"]

LEGAL_QRHE_MULTI_SOURCE_FILE = RETRIEVAL_DIR / "LegalQRHE_multi_source.json"
LEGAL_QRHE_HIENPHAP_DANSU_FILE = RETRIEVAL_DIR / "LegalQRHE_hienphap_dansu.json"
LEGAL_QRHE_ALL_FILE = RETRIEVAL_DIR / "LegalQRHE_all.json"

ROOT_DIR: C:\52300070_TMTHUYEN\LEARN_MORE\Apps\datn-legal-ai\backend


## Input truy vấn mẫu

In [35]:
inputs = [ 
    {
        "input": "Chương I của Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam năm 2013 quy định về vấn đề gì?",
        "output": {
            "part": None,
            "chapter": "I",
            "section": None,
            "subsection": None,
            "article": None,
            "clause": None,
            "point": None,
            "document": "Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam năm 2013"
        }
    },
    {
        "input": "Nội dung Điều 1 theo Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam năm 2013 là gì?",
        "output": {
            "part": None,
            "chapter": None,
            "section": None,
            "subsection": None,
            "article": "1",
            "clause": None,
            "point": None,
            "document": "Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam năm 2013"
        }
    },
]

## Cell chạy chạy Model hoặc Regex để lấy cấu trích trích dẫn

In [36]:
print("Chạy bóc tách cấu trúc")

Chạy bóc tách cấu trúc


## Kiểm trả tính hợp lệ của cấu trúc phân cấp đã trích xuất

## Tìm các Document liên quan trong Graph

## Tạo các node id từ cấu trúc phân cấp

## Tìm kiếm các node liên quan và các relationship